# Data Processing

## Import packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import *

## Load Data

Load data and select features with an SQL query through the duckdb.

Separate the features based on defined categories. This a crucial step because variables like `playername` usually have missing values because the database includes team stats. They should be treated separately. Furthermore, player stats will be engineered into different features later.

In [2]:
cols_dict = {'draft_cols': 
                ['gameid', 'date', 'teamid',
                 'firstPick', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5'],
            'player_cols': 
                ['gameid', 'date', 'teamid', 'playername', 'position', 'champion',
                 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim'],
            'team_cols': 
                ['gameid', 'date', 'teamid', 'teamname', 
                 'result', 'side', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 
                 'elders', 'opp_elders', 'firstherald', 'void_grubs', 'opp_void_grubs',
                 'firstbaron', 'barons', 'opp_barons', 'firsttower', 'towers', 'opp_towers'],
            'team&player_cols': 
                ['gameid', 'league', 'split', 'date', 'teamid',
                 'kills', 'assists', 'dpm', 'totalgold',
                 'doublekills', 'triplekills', 'quadrakills', 'pentakills',
                 'killsat10', 'assistsat10', 'opp_killsat10', 'opp_deathsat10',
                 'killsat15', 'assistsat15', 'opp_killsat15', 'opp_deathsat15',
                 'killsat20', 'assistsat20', 'opp_killsat20', 'opp_deathsat20',
                 'killsat25', 'assistsat25', 'opp_killsat25', 'opp_deathsat25'],
            'ingame_cols': 
                ['gameid', 'date', 'teamid',
                 'golddiffat10', 'xpdiffat10', 'golddiffat15', 'xpdiffat15', 
                 'golddiffat20', 'xpdiffat20', 'golddiffat25', 'xpdiffat25']}

for key in cols_dict.keys():
    cols_dict[key] = [f'"{col}"' for col in cols_dict[key]] # quote the cols for a smooth join

year_selected = "2026" # temporarily hardcoded df year

In [3]:
con = duckdb.connect("lol_predictors.duckdb")

df_dict = {}
for key in cols_dict.keys():
    cols = ", ".join([col for col in cols_dict[key]])
    df_dict[key] = con.execute(f"SELECT {cols} FROM df_{year_selected}").df()

con.close()

In [4]:
df_dict['player_cols'].info()

<class 'pandas.DataFrame'>
RangeIndex: 115032 entries, 0 to 115031
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   gameid            115032 non-null  str    
 1   date              115032 non-null  str    
 2   teamid            111786 non-null  str    
 3   playername        95860 non-null   str    
 4   position          115032 non-null  str    
 5   champion          95860 non-null   str    
 6   firstbloodkill    95780 non-null   float64
 7   firstbloodassist  95770 non-null   float64
 8   firstbloodvictim  95770 non-null   float64
dtypes: float64(3), str(6)
memory usage: 17.4 MB


Based on the EDA results, `year`, `patch`, and `gamelength` are removed because they have 0.00 correlation with the target variable `result`.
On the other hand, `towers`, `assists`, `assistsatX`, `goldspent`, `opp_deathsatX`, `golddiffat20`, and `xpdiffat20` are removed because they have high correlation with other variables, which may cause multicollinearity during modeling. This also applies to stats at the 25th minute mark as they correlate high with the stats at the 20th minute mark.

In [ ]:
# # Adjust list based on the features considered
# cols_to_include = ['gameid', 'league', 'split', 'date', 'teamname',
#                    'result', 'kills', 'deaths', 'dpm', 
#                    'damagetakenperminute', 'wardskilled', 'totalgold',
#                    'firstdragon', 'firstbaron', 'barons', 'opp_barons',
#                    'golddiffat10', 'xpdiffat10', 'killsat10', 'opp_killsat10',
#                    'golddiffat15', 'xpdiffat15', 'killsat15', 'opp_killsat15',
#                    'killsat20', 'opp_killsat20'
#                    ]

# db = 'lol_predictors.duckdb'
# df_selected = "df_2026" # temporarily hardcoded df name
# df = data_selection(df_selected, cols_to_include, db)

## Data Cleaning

For **data imputation**, only the feature `split` has missing values based on df_2014's EDA. We can use the previous value for each NaN since there's only two values, which are `Summer` and `Spring`, and they are sorted chronologically.

In [ ]:
df_dict['identifier_cols']['split'] = df_dict['identifier_cols']['split'].ffill()

We will **trim outliers** because even the feature `wardskilled` that has the highest outlier percentage only has `<2%` of outliers.

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns.tolist()

print(num_cols)

for col in num_cols:
    df = df[~detect_outliers(df, col).to_numpy()]

This concludes data cleaning of df_2014, which is the Lol Esports data on 2014 based on Oracle's Elixir. From the EDA, there are no categories in categorical features that needs fixing. 

## Post-Processing Data Audit

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
display(df.describe())

In [ ]:
data_profiling(df)

In [ ]:
# Limit numerical columns for description and visualization
numcols_to_drop = ['result', 'firstbaron', 'firstdragon']

# Limit categorical columns for description and visualization
catcols_to_drop = ['gameid', 'teamname']

In [ ]:
count_outliers(df, numcols_to_drop)

*Note that the IQR bounds are dynamically recalculated on each call. These remaining outliers are relative to the newly trimmed distribution. Since the new percentage of outliers is < 1%, it is negligible and safe to retain.*

In [ ]:
visualize_num_distribution(df, numcols_to_drop, nrows=8, ncols=4, figsize=(15, 30))

In [ ]:
# Visualization of the Splits Distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='split')
plt.title('DISTRIBUTION OF SPLITS', fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
visualize_outliers(df, numcols_to_drop, nrows=8, ncols=4, figsize=(15, 30))

In [ ]:
# Compute correlation matrix
co_mtx = df.corr(numeric_only=True)

# Plot correlation heatmap
plt.figure(figsize=(40, 40))
sns.heatmap(co_mtx, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('CORRELATION HEATMAP', fontsize=24, fontweight="bold")
plt.show()

In [ ]:
df.to_csv("df_processed.csv", index=False)